# BART-FaCT Preview: 基于层次化结构与校准式解码的摘要事实性增强

本 Notebook 演示 BART-FaCT 模型的三大创新模块及其可视化。

**核心思路：** 在 BART-Large-CNN（已在下游预训练好的摘要模型）基础上添加三个模块。

**创新点：**
1. **HSE** (Hierarchical Structure Encoding): 层次化结构编码 — 通过句子级Transformer捕获文档的层次结构（句子→段落→章节）
   - 灵感: Hierarchical Transformers (Liu & Lapata, EMNLP'19); Lost in the Middle (Liu et al., TACL'23)
2. **CFA** (Calibrated Faithfulness Attention): 校准式忠实度注意力 — 基于不确定性估计动态调节源文注意力
   - 灵感: DoLa Decoding (Chuang et al., ICLR'24); Context-Aware Decoding (Shi et al., ACL'24)
3. **CPO** (Contrastive Preference Optimization): 对比式偏好优化 — DPO风格偏好损失，用模型自生成负样本替代启发式扰动
   - 灵感: DPO (Rafailov et al., NeurIPS'23); Model-based Preference Optimization (EMNLP'24)

**Baseline:** BART-Large-CNN (已在 CNN/DailyMail 上预训练好的摘要模型)

**运行说明：** 从上到下依次执行每个 Cell 即可。

## 0. 安装依赖

In [ ]:
!pip install torch>=2.0.0 transformers>=4.35.0 datasets>=2.14.0 accelerate>=0.24.0
!pip install rouge-score>=0.1.2 bert-score>=0.3.13 nltk>=3.8
!pip install matplotlib>=3.7.0 seaborn>=0.12.0 pandas>=2.0.0 numpy>=1.24.0 scikit-learn>=1.3.0
!pip install tqdm>=4.65.0 sentencepiece>=0.1.99 protobuf>=4.21.0 tensorboard>=2.14.0

In [ ]:
import sys, os
if not os.path.exists('src') and not os.path.exists('../src'):
    !git clone https://github.com/zryshuaige/BART-FaCT.git /tmp/BART-FaCT
    !cp -r /tmp/BART-FaCT/src ./src 2>/dev/null || !cp -r /tmp/BART-FaCT/src ../src 2>/dev/null
for p in [os.path.join(os.getcwd(), 'src'), os.path.join(os.getcwd(), '..', 'src')]:
    if os.path.exists(p) and p not in sys.path: sys.path.insert(0, p)

IN_COLAB = 'google.colab' in str(get_ipython()) if hasattr(get_ipython(), '__module__') else False
HF_ENDPOINT = "https://huggingface.co" if IN_COLAB else "https://hf-mirror.com"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT
os.environ["HUGGINGFACE_HUB_TIMEOUT"] = "120"

import torch, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})
import datasets; datasets.config.HF_ENDPOINT = HF_ENDPOINT
import transformers; transformers.utils.hub.HF_ENDPOINT = HF_ENDPOINT
import huggingface_hub
if hasattr(huggingface_hub, "constants"): huggingface_hub.constants.HF_ENDPOINT = HF_ENDPOINT

from config import get_device, BARTFaCTConfig, ABLATION_CONFIGS, MODEL_CONFIGS
from data_utils import set_seed, load_arxiv_dataset
import data_utils as _du; _du._ensure_mirror = lambda: None

from models.bart_fact import BARTFaCTForConditionalGeneration
from models.hierarchical_structure import HierarchicalStructureEncoder, batch_detect_boundaries
from models.calibrated_attention import FaithfulnessCalibrator
from models.preference_loss import ContrastivePreferenceLoss, generate_context_free_summary
from visualization import *

device = get_device(); set_seed(42)
print(f'Device: {device}'); print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Environment ready!')

## Part 1: 数据加载
加载 arXiv 摘要数据集，取 120 条用于快速预览。

In [ ]:
ds = load_arxiv_dataset(max_samples=120)
test_key = 'test' if 'test' in ds else 'validation'
test_data = ds[test_key].select(range(min(20, len(ds[test_key]))))
sample = test_data[0]
print(f'训练集: {len(ds["train"])} / 测试集: {len(test_data)}')
print(f'论文长度: {len(sample["article"].split())} 词 / 摘要长度: {len(sample["abstract"].split())} 词')

## Part 2: 模块可视化（无需训练）

### 2.1 HSE: 层次化结构编码
**问题：** BART 将文档编码为扁平序列，丢失了句子→段落→章节的层次结构。
**解决：** HSE 通过句子边界检测 + 2层紧凑Transformer捕获文档的结构流。

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-cnn')
tokenizer.model_max_length = 1024

sample_text = test_data[0]['article']
boundary_mask = batch_detect_boundaries([sample_text], tokenizer, max_length=512)

# 统计句子边界
num_sentences = int(boundary_mask.sum().item())
total_tokens = boundary_mask.shape[1]
print(f'检测到 {num_sentences} 个句子边界 / {total_tokens} 个token')
print(f'平均句子长度: {total_tokens/max(num_sentences,1):.0f} tokens')

# 可视化句子边界位置
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(14, 2))
ax.imshow(boundary_mask.numpy(), aspect='auto', cmap='Blues')
ax.set_xlabel('Token Position'); ax.set_title('HSE: 句子边界检测 (蓝色=句子起点)')
plt.show()

In [ ]:
# HSE 参数量统计
hse = HierarchicalStructureEncoder(hidden_size=1024, num_heads=4, ffn_dim=256)
print(f'HSE 总参数量: {sum(p.numel() for p in hse.parameters()):,}')
print(f'  Sentence Transformer (2层): {sum(p.numel() for p in hse.sentence_transformer.parameters()):,}')
print(f'  Structure Gate: {sum(p.numel() for p in hse.structure_gate.parameters()):,}')
print(f'  (BART-large 基线: 400M, HSE 额外开销: {hse.param_count/4e5:.2f}%)')

### 2.2 CFA: 校准式忠实度注意力
**问题：** 标准交叉注意力对源文token等权关注，无法区分「检索事实」vs「流畅猜测」。
**解决：** CFA通过瓶颈校准网络估计不确定性，动态调节源文注意力的贡献比。

In [ ]:
cfa = FaithfulnessCalibrator(hidden_size=1024, bottleneck_dim=128)
print(f'CFA 每层参数量: {sum(p.numel() for p in cfa.parameters()):,}')
print(f'  编码器 (2048→128): {1024*2*128 + 128 + 128*2:,}')
print(f'  不确定性头 (128→64→1): {128*64 + 64 + 64*1 + 1:,}')
print(f'  忠实度投影 (128→1024): {128*1024 + 1024*2:,}')
print(f'  12层总计: ~{sum(p.numel() for p in cfa.parameters()) * 12 / 1e6:.1f}M')

# 模拟校准行为
np.random.seed(42)
num_layers, num_tokens = 12, 40
uncertainty_all = []
for layer in range(num_layers):
    # 高层通常不确定性更低（更忠实源文）
    base_uncertainty = 0.6 - 0.35 * (layer / num_layers)
    layer_unc = np.random.beta(base_uncertainty * 5, (1 - base_uncertainty) * 5, size=num_tokens)
    uncertainty_all.append(layer_unc)

print('\n各层平均不确定性 (越低=越自信=越忠实):')
for i, u in enumerate(uncertainty_all):
    bar = '█' * int((1 - u.mean()) * 30)
    print(f'  Layer {i:2d}: unc={u.mean():.3f} faith={1-u.mean():.3f} {bar}')

plot_fgca_gate_heatmap(
    [u.reshape(-1, 1).repeat(16, axis=1) for u in uncertainty_all],
    title='CFA: 各层不确定性热力图 (红=不确定/自主 绿=确定/忠实)',
)

### 2.3 CPO: 对比式偏好优化
**问题：** 交叉熵损失无法提供事实性偏好信号。InfoNCE+启发式扰动的负样本可能不匹配真实幻觉。
**解决：** CPO使用模型自生成的「无上下文摘要」作为非偏好响应，DPO风格损失拉向忠实、推离幻觉。

In [ ]:
cpo = ContrastivePreferenceLoss(hidden_size=1024, projection_dim=128)
print(f'CPO 投影头参数量: {sum(p.numel() for p in cpo.projector.parameters()):,}')
print(f'  Linear(1024→1024) + LN + GELU + Linear(1024→128)')
print(f'  DPO β: {cpo.beta}, CPO权重 λ: {cpo.alpha}')

# 模拟偏好间隔
np.random.seed(42); n = 200
pos_scores = np.random.normal(0.8, 0.15, n)  # 偏好响应分数
neg_scores = np.random.normal(0.3, 0.2, n)   # 非偏好响应分数

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pos_scores, bins=30, alpha=0.7, color='#27AE60', label=f'Preferred (ref) mean={pos_scores.mean():.3f}', density=True)
ax.hist(neg_scores, bins=30, alpha=0.7, color='#E74C3C', label=f'Dispreferred (no-ctx) mean={neg_scores.mean():.3f}', density=True)
ax.set_xlabel('Implicit Reward Score'); ax.set_ylabel('Density')
ax.set_title('CPO: 偏好 vs 非偏好响应分布'); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
margin = pos_scores.mean() - neg_scores.mean()
print(f'\n偏好间隔: {margin:.3f} (DPO损失将拉大此间隔)')

## Part 3: 快速训练与生成
使用 100 条训练数据快速训练 BART-FaCT（预览版，非最终性能）。

In [ ]:
from train import train_model; from config import TrainingConfig; import time
TRAINING_DONE = False
try:
    config = TrainingConfig(dataset_name='arxiv', model_name='bart-fact-full', max_samples=100,
        num_train_epochs=3, learning_rate=3e-5, per_device_train_batch_size=2,
        gradient_accumulation_steps=4, output_dir='./results/preview', fp16=(device.type=='cuda'))
    start = time.time()
    trainer, model, tokenizer = train_model(model_name='bart-fact-full', dataset_name='arxiv', max_samples=100, training_config=config)
    print(f'训练完成，用时 {(time.time()-start)/60:.1f} 分钟')
    param_info = model.get_trainable_params_summary()
    for k, v in param_info.items(): print(f'  {k}: {v:,}')
    TRAINING_DONE = True
except Exception as e:
    print(f'训练失败: {e} (GPU内存不足或CPU环境下可能遇到)\n将使用预训练BART-Large-CNN进行演示。')

In [ ]:
from evaluate import generate_summaries, compute_rouge
from transformers import AutoTokenizer, BartForConditionalGeneration

bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(device).eval()
bart_tokenizer = AutoTokenizer.from_pretrained('facebook/bart-large-cnn')

test_texts = [test_data[i]['article'] for i in range(3)]
references = [test_data[i]['abstract'] for i in range(3)]

bart_summaries = generate_summaries(model=bart_model, tokenizer=bart_tokenizer,
    texts=test_texts, max_input_length=1024, max_target_length=256, batch_size=1, device=device)

fact_summaries = bart_summaries  # fallback
if TRAINING_DONE:
    fact_summaries = generate_summaries(model=model, tokenizer=tokenizer,
        texts=test_texts, max_input_length=1024, max_target_length=256, batch_size=1, device=device, is_bart_fact=True)

for i in range(min(3, len(test_texts))):
    plot_summary_comparison(
        {'BART-Large-CNN (基线)': bart_summaries[i], 'BART-FaCT (完整)': fact_summaries[i]},
        reference=references[i], title=f'摘要对比 - 样本 {i+1}')
    br = compute_rouge([bart_summaries[i]], [references[i]])
    fr = compute_rouge([fact_summaries[i]], [references[i]])
    print(f'  BART: R1={br["rouge1"]["fmeasure"]:.3f} R2={br["rouge2"]["fmeasure"]:.3f} RL={br["rougeL"]["fmeasure"]:.3f}')
    print(f'  FaCT: R1={fr["rouge1"]["fmeasure"]:.3f} R2={fr["rouge2"]["fmeasure"]:.3f} RL={fr["rougeL"]["fmeasure"]:.3f}')

## Part 4: 消融实验
| 配置 | HSE | CFA | CPO |
|------|-----|------|-----|
| BART (Baseline) | ✗ | ✗ | ✗ |
| w/o HSE | ✗ | ✓ | ✓ |
| w/o CFA | ✓ | ✗ | ✓ |
| w/o CPO | ✓ | ✓ | ✗ |
| **BART-FaCT (Full)** | **✓** | **✓** | **✓** |

In [ ]:
from ablation import ABLATION_MODELS, run_single_ablation
ablation_results = {}
for name in ['bart_baseline', 'bart_fact_no_hse', 'bart_fact_no_cfa', 'bart_fact_no_cpo', 'bart_fact_full']:
    info = ABLATION_MODELS[name]; cfg = info['config']
    print(f'消融: {name} (HSE={cfg.use_hse} CFA={cfg.use_cfa} CPO={cfg.use_cpo})')
    try:
        result = run_single_ablation(ablation_name=name, dataset_name='arxiv', max_samples=100,
            num_test=5, output_dir='./results/preview_ablation', epochs=1, batch_size=2)
        ablation_results[name] = result
    except Exception as e:
        print(f'  失败: {e}'); ablation_results[name] = {'error': str(e)}
print(f'\n完成 {len(ablation_results)} 组消融实验')

## Part 5: 总结

In [ ]:
D = 1024  # BART-large hidden_size
hse_p = D*D*4*2*4 + D*256*2*4 + D*2*D + D*3  # ~2.7M
cfa_p = (D*2*128 + 128*2 + 128*64 + 64 + 64 + 128*D + D*2) * 12  # ~3.8M
cpo_p = D*D + D + D + D*128 + 128  # ~1.2M
bart_p = 400_000_000
print('='*60)
print('BART-FaCT 总结')
print('='*60)
print(f'\n1. HSE (层次化结构编码)     — 灵感: Hierarchical Transformers (EMNLP\'19)')
print(f'   句子检测→mean-pool→2层Transformer→门控广播')
print(f'   新增参数: ~{hse_p/1e6:.1f}M')
print(f'\n2. CFA (校准式忠实度注意力) — 灵感: DoLa (ICLR\'24) + CAD (ACL\'24)')
print(f'   不确定性估计→校准门控→调制源文注意力贡献')
print(f'   新增参数: ~{cfa_p/1e6:.1f}M (12层总计)')
print(f'\n3. CPO (对比式偏好优化)     — 灵感: DPO (NeurIPS\'23)')
print(f'   自生成偏好对→DPO风格损失→拉向忠实、推离幻觉')
print(f'   新增参数: ~{cpo_p/1e6:.1f}M')
print(f'\n模型总参数: ~{(bart_p + hse_p + cfa_p + cpo_p)/1e6:.0f}M (BART: 400M)')
print(f'模块额外开销: ~{(hse_p + cfa_p + cpo_p)/1e6:.1f}M ({(hse_p+cfa_p+cpo_p)/bart_p*100:.1f}%)')
print(f'\n消融实验: bart_baseline → no_hse → no_cfa → no_cpo → full')
print(f'完整实验: python src/run_experiments.py --mode full --max_samples 5000')